# Clinicopathological and Molecular Characteristics of Second Primary Colorectal Cancer in Cancer Survivors including MSI-H Status and Anatomical Distribution Exploration with `mlcroissant`
This notebook demonstrates loading and exploring the FAIR² colorectal cancer survivor dataset using the `mlcroissant` library.

### Dataset Source
The dataset source is specified by a Croissant schema URL. All entities (record sets, fields, columns, etc.) are referenced via their `@id` per Croissant conventions.

In [ ]:
# Ensure `mlcroissant` library is installed
!pip install mlcroissant

## 1. Data Loading
Load metadata and records from the dataset using `mlcroissant`.

In [ ]:
import mlcroissant as mlc
import pandas as pd
import json

# Define the Croissant schema URL
croissant_url = "https://sen.science/doi/10.71728/senscience.qs2f-h81p/fair2.json"

# Load the dataset using mlcroissant
dataset = mlc.Dataset(croissant_url)

# Access metadata and print a summary
metadata = dataset.metadata.to_json()
print("Dataset Title:", metadata.get('name', 'N/A'))
print("Description:", metadata.get('description', 'N/A'))
print("Citation:", metadata.get('citeAs', 'N/A'))
print("Authors (IDs):", metadata.get('author', []))

## 2. Data Overview
Review available record sets, fields, and their `@id`s.

In [ ]:
# List available record sets, fields, and their @ids
print("\nAvailable Record Sets:")
record_sets = list(dataset.record_sets)
for rs in record_sets:
    print(f" • @id: {rs['@id']} | name: {rs.get('name','<no name>')}")
    fields = rs.get('field', [])
    # fields can be dict or list
    if isinstance(fields, dict):
        fields = [fields]
    if fields:
        print("   Fields:")
        for f in fields:
            if isinstance(f, dict):
                print(f"    - @id: {f.get('@id','')} | name: {f.get('name','')}")
            else:
                print(f"    - @id: {f}")
    else:
        print("   (No fields listed)")

## 3. Data Extraction
Load data from each record set into a DataFrame for analysis.

We use the record set `@id` and its field IDs as discovered above.

In [ ]:
from collections import defaultdict

# Build a list of record set @ids
record_set_ids = [rs['@id'] for rs in dataset.record_sets]
dataframes = {}
print("\nLoading records for each record set:")
for record_set_id in record_set_ids:
    # Extract all records for the given record set
    records = list(dataset.records(record_set=record_set_id))
    if records:
        df = pd.DataFrame(records)
        dataframes[record_set_id] = df
        print(f"\u2714 Loaded {len(df)} records for {record_set_id} with columns: {df.columns.tolist()}")
    else:
        print(f"\u2718 No records found for {record_set_id}")

# For demonstration, pick the first record set with data
if len(dataframes) > 0:
    main_record_set_id = list(dataframes.keys())[0]
    print(f"\nPreview of records in {main_record_set_id}:\n")
    display(dataframes[main_record_set_id].head())
    print(f"Available columns/fields for {main_record_set_id}: {dataframes[main_record_set_id].columns.tolist()}")
else:
    main_record_set_id = None

## 4. Exploratory Data Analysis (EDA)
Apply common data processing steps, such as filtering records based on specific criteria, normalizing numeric fields, and grouping data by key attributes.

Below, we demonstrate filtering and normalization for a numeric field, and grouping if a groupable field exists.

In [ ]:
import numpy as np

# If no record sets were loaded, skip this section
if main_record_set_id is None:
    print("No data available for EDA.")
else:
    df = dataframes[main_record_set_id]

    # Search for a numeric field in the columns
    numeric_field = None
    for col in df.columns:
        if np.issubdtype(df[col].dropna().apply(type), np.number).any():
            numeric_field = col
            break
    # If not found, guess: look for col names containing 'age'/'year'
    if numeric_field is None:
        for col in df.columns:
            if 'age' in col.lower() or 'year' in col.lower():
                numeric_field = col
                break

    if numeric_field is not None:
        print(f"Numeric field selected: {numeric_field}")
        # Convert to numeric if needed
        df[numeric_field] = pd.to_numeric(df[numeric_field], errors='coerce')
        # Set a threshold (here, arbitrarily median)
        threshold = df[numeric_field].median()
        filtered_df = df[df[numeric_field] > threshold].copy()
        print(f"Filtered records with {numeric_field} > {threshold}:")
        display(filtered_df.head())

        # Normalize
        norm_col = f"{numeric_field}_normalized"
        filtered_df[norm_col] = (filtered_df[numeric_field] - filtered_df[numeric_field].mean()) / filtered_df[numeric_field].std()
        print(f"\nNormalized {numeric_field} for filtered records:")
        display(filtered_df[[numeric_field, norm_col]].head())

        # Attempt to group by a categorical field
        group_field = None
        # Pick first object-type field that is not the numeric_field
        obj_types = [col for col in filtered_df.columns if filtered_df[col].dtype == object and col != numeric_field]
        if obj_types:
            group_field = obj_types[0]
        if group_field is not None:
            grouped_df = filtered_df.groupby(group_field)[numeric_field].mean().reset_index()
            print(f"\nGrouped mean of {numeric_field} by {group_field}:")
            display(grouped_df.head())
        else:
            print("No group field found for grouping.")
    else:
        print("No numeric field found for EDA.")

## 5. Visualization
Visualize a field's distribution and a relationship between two fields, if appropriate numeric/categorical columns are available.

In [ ]:
import matplotlib.pyplot as plt
import seaborn as sns

if main_record_set_id is not None and numeric_field is not None:
    plt.figure(figsize=(8, 5))
    sns.histplot(df[numeric_field].dropna(), bins=15, kde=True)
    plt.title(f"Distribution of {numeric_field}")
    plt.xlabel(numeric_field)
    plt.ylabel("Count")
    plt.show()
    
    # If we have a group_field, make a boxplot
    if group_field is not None:
        plt.figure(figsize=(10, 6))
        sns.boxplot(x=filtered_df[group_field], y=filtered_df[numeric_field])
        plt.title(f"{numeric_field} by {group_field}")
        plt.xlabel(group_field)
        plt.ylabel(numeric_field)
        plt.xticks(rotation=45)
        plt.show()


## 6. Conclusion
In this notebook, we demonstrated how to load, review, and perform initial exploration on the FAIR² colorectal cancer survivor dataset using the `mlcroissant` library. We accessed schema-driven data, examined available record sets and fields by `@id`, and loaded records for further analysis. Using common data processing (filtering, normalization, grouping) and simple plots, we established a reproducible workflow to accelerate medical and scientific analysis.

Further work could include more sophisticated preprocessing, model development, or exporting selected records for downstream use.